In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

import joblib

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Loading my datasets

train = pd.read_csv('../data/Train.csv')
test = pd.read_csv('../data/Test.csv')

print(train.shape)
print(test.shape)

In [ ]:
train.head(5)

In [ ]:
# checking out my columns types

train.info()

In [ ]:
# STATISTICAL SUMMARY
train.describe()

In [ ]:
# CHECKING CLASS DISTRIBUTION
train['Target'].value_counts()

In [ ]:
# VISUALIZING MY TARGET
sns.countplot(
    x='Target',
    data=train
)

plt.title('Target Distribution')

plt.show()

In [ ]:
# CHECK MISSING VALUES
missing_values = train.isnull().sum()

missing_values = missing_values[
    missing_values > 0
]

missing_values.sort_values(
    ascending=False
)

In [ ]:
# IDENTIFY FEATURE TYPES
categorical_cols = train.select_dtypes(
    include=['object']
).columns

numerical_cols = train.select_dtypes(
    exclude=['object']
).columns

print("Categorical Columns:")
print(categorical_cols)

print()

print("Numerical Columns:")
print(numerical_cols)

In [ ]:
# SPLIT FEATURES AND TARGET
TARGET = 'Target'

X = train.drop(columns=[TARGET])

y = train[TARGET]

In [ ]:
# ENCODE CATEGORICAL VARIABLES

label_encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    combined_data = pd.concat(
        [X[col], test[col]],
        axis=0
    ).astype(str)

    le.fit(combined_data)

    X[col] = le.transform(
        X[col].astype(str)
    )

    test[col] = le.transform(
        test[col].astype(str)
    )

    label_encoders[col] = le

In [ ]:
# HANDLE MISSING VALUES
imputer = SimpleImputer(
    strategy='median'
)

X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

test = pd.DataFrame(
    imputer.transform(test),
    columns=test.columns
)

In [ ]:
# TRAIN/VALIDATION SPLIT
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
# HANDLE IMBALANCE USING SMOTE
smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(
    X_train,
    y_train
)

In [ ]:
# CHECKING NEW DISTRIBUTION
y_resampled.value_counts()

In [ ]:
# BUILDING BASELINE MODEL
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_model.fit(
    X_resampled,
    y_resampled
)

In [ ]:
# MAKE PREDICTIONS
rf_preds = rf_model.predict(X_valid)

rf_probs = rf_model.predict_proba(
    X_valid
)[:, 1]

In [ ]:
# EVALUATING THE  MODEL
print("Accuracy:", accuracy_score(y_valid, rf_preds))
# print("Precision:", precision_score(y_valid, rf_preds))
# print("Recall:", recall_score(y_valid, rf_preds))
# print("F1-Score:", f1_score(y_valid, rf_preds))

In [ ]:
# ROC AUC SCORE
roc_auc = roc_auc_score(y_valid, rf_probs)
roc_auc

In [ ]:
# Classification Report
print(
    classification_report(
        y_valid,
        rf_preds
    )
)

In [ ]:
# CONFUSION MATRIX
cm = confusion_matrix(
    y_valid,
    rf_preds
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='YlGnBu'
)

plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.title('Confusion Matrix')

plt.show()

In [ ]:
#  BUILD ADVANCED XGBOOST MODEL

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(
    X_resampled,
    y_resampled
)

In [ ]:
# EVALUATE XGBOOST

xgb_preds = xgb_model.predict(
    X_valid
)

xgb_probs = xgb_model.predict_proba(
    X_valid
)[:,1]

In [ ]:
# Metrics
print("Accuracy:",
      accuracy_score(y_valid, xgb_preds))

print()

print("ROC AUC:",
      roc_auc_score(y_valid, xgb_probs))

print()

print(classification_report(
    y_valid,
    xgb_preds
))

In [ ]:
# FEATURE IMPORTANCE

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

importance.head(20)

In [ ]:
# VISUALIZE IMPORTANT FEATURES
plt.figure(figsize=(10,8))

sns.barplot(
    x='Importance',
    y='Feature',
    data=importance.head(20)
)

plt.title(
    'Top 20 Important Features'
)

plt.show()

In [ ]:
# CROSS VALIDATION
kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    xgb_model,
    X,
    y,
    cv=kfold,
    scoring='roc_auc'
)

print(scores)

print()

print("Mean ROC AUC:",
      scores.mean())

In [ ]:
smote = SMOTE(random_state=42)

X_final, y_final = smote.fit_resample(
    X,
    y
)

final_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

final_model.fit(
    X_final,
    y_final
)

In [ ]:
# XGBOOST PREDICTIONS

xgb_preds = xgb_model.predict(
    X_valid
)

xgb_probs = xgb_model.predict_proba(
    X_valid
)[:,1]

xgb_preds

In [ ]:
# XGBOOST EVALUATION


print(
    "Accuracy:",
    accuracy_score(
        y_valid,
        xgb_preds
    )
)

print()

print(
    "ROC AUC:",
    roc_auc_score(
        y_valid,
        xgb_probs
    )
)

print()

print(
    classification_report(
        y_valid,
        xgb_preds
    )
)

In [ ]:
# =========================
# CONFUSION MATRIX
# =========================

cm = confusion_matrix(
    y_valid,
    xgb_preds
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel('Predicted')

plt.ylabel('Actual')

plt.title('XGBoost Confusion Matrix')

plt.show()

In [ ]:
# TESTING PREDICTIONS
test_predictions = xgb_model.predict_proba(
    test
)[:,1]

print(test_predictions[:10])

In [ ]:
len(test_predictions)

In [ ]:
len(test)

In [ ]:
submission = pd.read_csv(
    '../data/SampleSubmission.csv'
)

submission['Target'] = test_predictions

submission.head()

In [ ]:
submission.to_csv(
    'submission.csv',
    index=False
)

print("Submission File Saved Successfully!")

In [ ]:
import joblib

joblib.dump(
    xgb_model,
    '../models/loan_default_xgb_model.pkl'
)

print("Model Saved Successfully")

In [ ]:
loaded_model = joblib.load(
    '../models/loan_default_xgb_model.pkl'
)

print("Model Loaded Successfully")

In [ ]:
# =========================
# SINGLE CUSTOMER PREDICTION
# =========================

sample_customer = X.iloc[[0]]

prediction = loaded_model.predict(
    sample_customer
)

probability = loaded_model.predict_proba(
    sample_customer
)

print("Prediction:", prediction[0])

print(
    "Default Probability:",
    probability[:,1][0]
)

In [ ]:
# =========================
# FEATURE ENGINEERING
# =========================

# Example features

# Only run if columns exist

# train['loan_income_ratio'] = (
#     train['loan_amount'] /
#     train['income']
# )

# train['payment_income_ratio'] = (
#     train['monthly_payment'] /
#     train['income']
# )

print("Feature Engineering Section Ready")

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [4, 6],
    'learning_rate': [0.01, 0.05],
    'n_estimators': [200, 500]
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    ),
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    verbose=1
)

grid_search.fit(
    X_resampled,
    y_resampled
)

print(grid_search.best_params_)

In [ ]:
import shap

explainer = shap.TreeExplainer(
    xgb_model
)

shap_values = explainer.shap_values(
    X_valid
)

shap.summary_plot(
    shap_values,
    X_valid
)

In [ ]:
custom_preds = (xgb_probs >= 0.3).astype(int)

In [ ]:
# print(
#     classification_report(
#         y_valid,
#         custom_preds
#     )
# )

# =========================
# BEST THRESHOLD PREDICTIONS
# =========================

best_threshold = 0.2

best_preds = (
    xgb_probs >= best_threshold
).astype(int)

print(
    classification_report(
        y_valid,
        best_preds
    )
)

In [ ]:
# =========================
# RELOAD CLEAN DATA
# =========================

train_fe = train.copy()

test_fe = test.copy()

print("Feature Engineering Dataset Ready")

In [ ]:
# =========================
# CONVERT DATE COLUMNS
# =========================

date_cols = [
    'date_approved',
    'date_disbursed',
    'first_payment_due',
    'maturity_date',
    'client_dob'
]

for col in date_cols:

    train_fe[col] = pd.to_datetime(
        train_fe[col],
        dayfirst=True,
        errors='coerce'
    )

    test_fe[col] = pd.to_datetime(
        test_fe[col],
        dayfirst=True,
        errors='coerce'
    )

print("Date Conversion Completed")

In [ ]:
# =========================
# CUSTOMER AGE
# =========================

train_fe['customer_age'] = (
    pd.Timestamp('today') -
    train_fe['client_dob']
).dt.days // 365

test_fe['customer_age'] = (
    pd.Timestamp('today') -
    test_fe['client_dob']
).dt.days // 365

print("Customer Age Feature Created")

In [ ]:
# =========================
# LOAN DURATION
# =========================

train_fe['loan_duration_days'] = (
    train_fe['maturity_date'] -
    train_fe['date_disbursed']
).dt.days

test_fe['loan_duration_days'] = (
    test_fe['maturity_date'] -
    test_fe['date_disbursed']
).dt.days

print("Loan Duration Feature Created")

In [ ]:
# =========================
# APPROVAL DELAY
# =========================

train_fe['approval_delay_days'] = (
    train_fe['date_disbursed'] -
    train_fe['date_approved']
).dt.days

test_fe['approval_delay_days'] = (
    test_fe['date_disbursed'] -
    test_fe['date_approved']
).dt.days

print("Approval Delay Feature Created")

In [ ]:
# =========================
# LOAN TO INCOME RATIO
# =========================

train_fe['loan_income_ratio'] = (
    train_fe['amount_usd'] /
    (train_fe['monthly_income_usd'] + 1)
)

test_fe['loan_income_ratio'] = (
    test_fe['amount_usd'] /
    (test_fe['monthly_income_usd'] + 1)
)

print("Loan Income Ratio Created")

In [ ]:
# =========================
# DEBT TO INCOME RATIO
# =========================

train_fe['debt_income_ratio'] = (
    train_fe['existing_obligations'] /
    (train_fe['monthly_income_usd'] + 1)
)

test_fe['debt_income_ratio'] = (
    test_fe['existing_obligations'] /
    (test_fe['monthly_income_usd'] + 1)
)

print("Debt Income Ratio Created")

In [ ]:
# =========================
# ESTIMATED MONTHLY PAYMENT
# =========================

train_fe['estimated_monthly_payment'] = (
    train_fe['amount_usd'] /
    (train_fe['term_months'] + 1)
)

test_fe['estimated_monthly_payment'] = (
    test_fe['amount_usd'] /
    (test_fe['term_months'] + 1)
)

print("Estimated Monthly Payment Created")

In [ ]:
# =========================
# PAYMENT STRESS
# =========================

train_fe['payment_stress_ratio'] = (
    train_fe['estimated_monthly_payment'] /
    (train_fe['monthly_income_usd'] + 1)
)

test_fe['payment_stress_ratio'] = (
    test_fe['estimated_monthly_payment'] /
    (test_fe['monthly_income_usd'] + 1)
)

print("Payment Stress Ratio Created")

In [ ]:
# =========================
# EMPLOYMENT STABILITY
# =========================

train_fe['employment_stability'] = (
    train_fe['months_at_employer'] / 12
)

test_fe['employment_stability'] = (
    test_fe['months_at_employer'] / 12
)

print("Employment Stability Created")

In [ ]:
# =========================
# DROP RAW DATE COLUMNS
# =========================

train_fe = train_fe.drop(
    columns=date_cols
)

test_fe = test_fe.drop(
    columns=date_cols
)

print("Raw Date Columns Dropped")

In [ ]:
# =========================
# SPLIT FEATURES & TARGET
# =========================

TARGET = 'Target'

X = train_fe.drop(
    columns=[TARGET]
)

y = train_fe[TARGET]

print(X.shape)

In [ ]:
categorical_cols = X.select_dtypes(
    include=['object']
).columns

print(categorical_cols)

In [ ]:
# =========================
# LABEL ENCODING
# =========================

label_encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    combined = pd.concat(
        [X[col], test_fe[col]],
        axis=0
    ).astype(str)

    le.fit(combined)

    X[col] = le.transform(
        X[col].astype(str)
    )

    test_fe[col] = le.transform(
        test_fe[col].astype(str)
    )

print("Encoding Completed")

In [ ]:
# =========================
# HANDLE MISSING VALUES
# =========================

imputer = SimpleImputer(
    strategy='median'
)

X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

test_fe = pd.DataFrame(
    imputer.transform(test_fe),
    columns=test_fe.columns
)

print("Missing Values Handled")

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(
    X_train,
    y_train
)

In [ ]:
improved_xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=4,
    min_child_weight=3,
    gamma=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=3,
    random_state=42,
    eval_metric='logloss'
)

improved_xgb.fit(
    X_resampled,
    y_resampled
)

print("Improved Model Trained")

In [ ]:
improved_probs = improved_xgb.predict_proba(
    X_valid
)[:,1]

improved_preds = (
    improved_probs >= 0.2
).astype(int)

print(
    classification_report(
        y_valid,
        improved_preds
    )
)

print()

print(
    "ROC AUC:",
    roc_auc_score(
        y_valid,
        improved_probs
    )
)

In [ ]:
# =========================
# FINAL CALIBRATED PREDICTIONS
# =========================

best_threshold = 0.4

final_preds = (
    improved_probs >= best_threshold
).astype(int)

print(
    classification_report(
        y_valid,
        final_preds
    )
)

print()

print(
    "ROC AUC:",
    roc_auc_score(
        y_valid,
        improved_probs
    )
)

In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    eval_metric='AUC',
    verbose=100,
    random_seed=42
)

cat_model.fit(
    X_resampled,
    y_resampled
)

print("CatBoost Training Complete")

In [ ]:
cat_probs = cat_model.predict_proba(
    X_valid
)[:,1]

cat_preds = (
    cat_probs >= 0.4
).astype(int)

print(
    classification_report(
        y_valid,
        cat_preds
    )
)

print()

print(
    "ROC AUC:",
    roc_auc_score(
        y_valid,
        cat_probs
    )
)